# 08_07 Results Consolidation – Configurable Pipeline

## Purpose

This notebook consolidates final results from the full EES pipeline.

It is configurable across:

- German Credit
- Pima Diabetes
- COMPAS

Only the `ACTIVE_DATASET` value should be changed.

## What This Notebook Consolidates

This notebook brings together:

1. Baseline model performance from `03`
2. Fidelity scores from `08_02`
3. Stability scores from `08_03`
4. Actionability scores from `08_04`
5. Regulatory alignment scores from `08_05`
6. Final EES/AHP scores from `08_06`
7. AHP sensitivity results from `08_06`
8. AHP ranking robustness summary from `08_06`
9. AHP consistency diagnostics from `08_06`

## Updated EES Design

### Fidelity

```text
Fidelity =
1/3 SHAP Reconstruction Fidelity
+ 1/3 LIME Local Surrogate Fidelity
+ 1/3 SHAP-LIME Agreement
```

### Actionability

```text
Actionability =
1/3 Availability
+ 1/3 Proximity
+ 1/3 Sparsity
```

### Final Primary EES

```text
EES =
0.25 Fidelity
+ 0.25 Stability
+ 0.25 Actionability
+ 0.25 Regulatory Alignment
```

The primary EES score is the AHP-derived equal-weight score.  
Alternative AHP-derived weighting schemes are consolidated for sensitivity analysis.

## Final Consolidation Outputs

This notebook creates:

- `final_consolidated_results.csv`
- `publication_ready_summary.csv`
- `model_ranking_summary.csv`
- `ees_component_score_matrix.csv`
- `ees_sensitivity_summary.csv`
- `ees_ranking_robustness_summary.csv`
- `ahp_diagnostics_summary.csv`
- `ahp_scheme_weights_summary.csv`
- `methodology_audit_table.csv`
- consolidated Excel workbook

These outputs support dissertation writing, supervisor review, and journal article reporting.

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# ACTIVE_DATASET = "german_credit"
ACTIVE_DATASET = "pima_diabetes"
# ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "results_folder": "german_credit",
        "baseline_folder": "03_baseline_modeling_german_credit",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_german_credit",
        "stability_folder": "08_03_ees_framework_stability_score_german_credit",
        "actionability_folder": "08_04_ees_framework_actionability_score_german_credit",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_german_credit",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_german_credit",
        "consolidation_folder": "08_07_results_consolidation_german_credit"
    },
    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "results_folder": "pima_diabetes",
        "baseline_folder": "03_baseline_modeling_pima_diabetes",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_pima_diabetes",
        "stability_folder": "08_03_ees_framework_stability_score_pima_diabetes",
        "actionability_folder": "08_04_ees_framework_actionability_score_pima_diabetes",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_pima_diabetes",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_pima_diabetes",
        "consolidation_folder": "08_07_results_consolidation_pima_diabetes"
    },
    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "results_folder": "compas",
        "baseline_folder": "03_baseline_modeling_compas",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_compas",
        "stability_folder": "08_03_ees_framework_stability_score_compas",
        "actionability_folder": "08_04_ees_framework_actionability_score_compas",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_compas",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_compas",
        "consolidation_folder": "08_07_results_consolidation_compas"
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])

Active dataset: Pima Diabetes
Dataset name: pima_diabetes


In [2]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 400)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["consolidation_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

BASELINE_RESULTS_PATH = RESULTS_ROOT / CONFIG["baseline_folder"]
BASELINE_TABLES_PATH = BASELINE_RESULTS_PATH / "tables"
BASELINE_DATASETS_PATH = BASELINE_RESULTS_PATH / "datasets"

FIDELITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["fidelity_folder"]
FIDELITY_TABLES_PATH = FIDELITY_RESULTS_PATH / "tables"
FIDELITY_DATASETS_PATH = FIDELITY_RESULTS_PATH / "datasets"

STABILITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["stability_folder"]
STABILITY_TABLES_PATH = STABILITY_RESULTS_PATH / "tables"
STABILITY_DATASETS_PATH = STABILITY_RESULTS_PATH / "datasets"

ACTIONABILITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["actionability_folder"]
ACTIONABILITY_TABLES_PATH = ACTIONABILITY_RESULTS_PATH / "tables"
ACTIONABILITY_DATASETS_PATH = ACTIONABILITY_RESULTS_PATH / "datasets"

REGULATORY_RESULTS_PATH = RESULTS_ROOT / CONFIG["regulatory_folder"]
REGULATORY_TABLES_PATH = REGULATORY_RESULTS_PATH / "tables"
REGULATORY_DATASETS_PATH = REGULATORY_RESULTS_PATH / "datasets"

AHP_RESULTS_PATH = RESULTS_ROOT / CONFIG["ahp_folder"]
AHP_TABLES_PATH = AHP_RESULTS_PATH / "tables"
AHP_DATASETS_PATH = AHP_RESULTS_PATH / "datasets"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("Consolidation output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes
Consolidation output path: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes


In [4]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)
        if path.exists():
            print(f"{label} found at: {path}")
            return path

    if required:
        checked = "\\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\\n{checked}")

    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    print(f"Saved: {output_path}")
    return output_path


def safe_cols(df, cols):
    return [col for col in cols if col in df.columns]


def add_dataset_metadata(df):
    if df.empty:
        return df
    out = df.copy()
    if "dataset_name" not in out.columns:
        out.insert(0, "dataset_name", DATASET_NAME)
    if "dataset_label" not in out.columns:
        out.insert(1, "dataset_label", CONFIG["dataset_label"])
    return out

## 4. Load All Final Pipeline Outputs

In [5]:
# =========================================================
# 4. Load All Final Pipeline Outputs
# =========================================================

baseline_file = find_first_existing_file(
    [
        BASELINE_DATASETS_PATH / "baseline_model_performance.csv",
        BASELINE_TABLES_PATH / "baseline_model_performance.csv",
        BASELINE_DATASETS_PATH / "model_performance.csv",
        BASELINE_TABLES_PATH / "model_performance.csv"
    ],
    "Baseline model performance"
)

fidelity_file = find_first_existing_file(
    [
        FIDELITY_DATASETS_PATH / "ees_fidelity_model_scores.csv",
        FIDELITY_TABLES_PATH / "ees_fidelity_model_scores.csv",
        FIDELITY_DATASETS_PATH / "fidelity_score_summary.csv",
        FIDELITY_TABLES_PATH / "fidelity_score_summary.csv"
    ],
    "Fidelity model scores"
)

stability_file = find_first_existing_file(
    [
        STABILITY_DATASETS_PATH / "ees_stability_model_scores.csv",
        STABILITY_TABLES_PATH / "ees_stability_model_scores.csv",
        STABILITY_DATASETS_PATH / "stability_score_summary.csv",
        STABILITY_TABLES_PATH / "stability_score_summary.csv"
    ],
    "Stability model scores"
)

actionability_file = find_first_existing_file(
    [
        ACTIONABILITY_DATASETS_PATH / "ees_actionability_model_scores.csv",
        ACTIONABILITY_TABLES_PATH / "ees_actionability_model_scores.csv",
        ACTIONABILITY_DATASETS_PATH / "actionability_score_summary.csv",
        ACTIONABILITY_TABLES_PATH / "actionability_score_summary.csv"
    ],
    "Actionability model scores"
)

regulatory_file = find_first_existing_file(
    [
        REGULATORY_DATASETS_PATH / "ees_regulatory_alignment_model_scores.csv",
        REGULATORY_TABLES_PATH / "ees_regulatory_alignment_model_scores.csv",
        REGULATORY_DATASETS_PATH / "regulatory_alignment_score_summary.csv",
        REGULATORY_TABLES_PATH / "regulatory_alignment_score_summary.csv"
    ],
    "Regulatory alignment model scores"
)

ees_file = find_first_existing_file(
    [
        AHP_DATASETS_PATH / "ees_final_model_scores.csv",
        AHP_TABLES_PATH / "ees_final_model_scores.csv",
        AHP_DATASETS_PATH / "ees_ahp_score_summary.csv",
        AHP_TABLES_PATH / "ees_ahp_score_summary.csv"
    ],
    "Final EES model scores"
)

component_contribution_file = find_first_existing_file(
    [
        AHP_DATASETS_PATH / "ees_component_contribution_long.csv",
        AHP_TABLES_PATH / "ees_component_contribution_long.csv"
    ],
    "EES component contribution long",
    required=False
)

weighting_sensitivity_file = find_first_existing_file(
    [
        AHP_DATASETS_PATH / "ees_weighting_sensitivity_long.csv",
        AHP_TABLES_PATH / "ees_weighting_sensitivity_long.csv",
        AHP_DATASETS_PATH / "ees_weighting_sensitivity.csv",
        AHP_TABLES_PATH / "ees_weighting_sensitivity.csv"
    ],
    "EES weighting sensitivity",
    required=False
)

ranking_robustness_file = find_first_existing_file(
    [
        AHP_DATASETS_PATH / "ees_ranking_robustness_summary.csv",
        AHP_TABLES_PATH / "ees_ranking_robustness_summary.csv"
    ],
    "EES ranking robustness summary",
    required=False
)

ahp_scheme_weights_file = find_first_existing_file(
    [
        AHP_DATASETS_PATH / "ees_ahp_scheme_weights.csv",
        AHP_TABLES_PATH / "ees_ahp_scheme_weights.csv"
    ],
    "AHP scheme weights",
    required=False
)

ahp_diagnostics_file = find_first_existing_file(
    [
        AHP_DATASETS_PATH / "ees_ahp_scheme_consistency_diagnostics.csv",
        AHP_TABLES_PATH / "ees_ahp_scheme_consistency_diagnostics.csv"
    ],
    "AHP scheme consistency diagnostics",
    required=False
)

ahp_pairwise_matrices_file = find_first_existing_file(
    [
        AHP_DATASETS_PATH / "ees_ahp_pairwise_matrices.csv",
        AHP_TABLES_PATH / "ees_ahp_pairwise_matrices.csv"
    ],
    "AHP pairwise matrices",
    required=False
)

ahp_scheme_metadata_file = find_first_existing_file(
    [
        AHP_DATASETS_PATH / "ees_ahp_scheme_metadata.csv",
        AHP_TABLES_PATH / "ees_ahp_scheme_metadata.csv"
    ],
    "AHP scheme metadata",
    required=False
)

baseline_df = read_csv_safely(baseline_file, "Baseline model performance")
fidelity_df = read_csv_safely(fidelity_file, "Fidelity model scores")
stability_df = read_csv_safely(stability_file, "Stability model scores")
actionability_df = read_csv_safely(actionability_file, "Actionability model scores")
regulatory_df = read_csv_safely(regulatory_file, "Regulatory alignment model scores")
ees_df = read_csv_safely(ees_file, "Final EES model scores")

component_contribution_df = (
    read_csv_safely(component_contribution_file, "EES component contribution long", required=False)
    if component_contribution_file else pd.DataFrame()
)

weighting_sensitivity_df = (
    read_csv_safely(weighting_sensitivity_file, "EES weighting sensitivity", required=False)
    if weighting_sensitivity_file else pd.DataFrame()
)

ranking_robustness_df = (
    read_csv_safely(ranking_robustness_file, "EES ranking robustness summary", required=False)
    if ranking_robustness_file else pd.DataFrame()
)

ahp_scheme_weights_df = (
    read_csv_safely(ahp_scheme_weights_file, "AHP scheme weights", required=False)
    if ahp_scheme_weights_file else pd.DataFrame()
)

ahp_diagnostics_df = (
    read_csv_safely(ahp_diagnostics_file, "AHP diagnostics", required=False)
    if ahp_diagnostics_file else pd.DataFrame()
)

ahp_pairwise_matrices_df = (
    read_csv_safely(ahp_pairwise_matrices_file, "AHP pairwise matrices", required=False)
    if ahp_pairwise_matrices_file else pd.DataFrame()
)

ahp_scheme_metadata_df = (
    read_csv_safely(ahp_scheme_metadata_file, "AHP scheme metadata", required=False)
    if ahp_scheme_metadata_file else pd.DataFrame()
)

print("baseline_df:", baseline_df.shape)
print("fidelity_df:", fidelity_df.shape)
print("stability_df:", stability_df.shape)
print("actionability_df:", actionability_df.shape)
print("regulatory_df:", regulatory_df.shape)
print("ees_df:", ees_df.shape)
print("component_contribution_df:", component_contribution_df.shape)
print("weighting_sensitivity_df:", weighting_sensitivity_df.shape)
print("ranking_robustness_df:", ranking_robustness_df.shape)
print("ahp_scheme_weights_df:", ahp_scheme_weights_df.shape)
print("ahp_diagnostics_df:", ahp_diagnostics_df.shape)
print("ahp_pairwise_matrices_df:", ahp_pairwise_matrices_df.shape)
print("ahp_scheme_metadata_df:", ahp_scheme_metadata_df.shape)

Baseline model performance found at: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/03_baseline_modeling_pima_diabetes/datasets/baseline_model_performance.csv
Fidelity model scores found at: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_02_ees_framework_fidelity_score_pima_diabetes/datasets/ees_fidelity_model_scores.csv
Stability model scores found at: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_03_ees_framework_stability_score_pima_diabetes/datasets/ees_stability_model_scores.csv
Actionability model scores found at: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_04_ees_framework_actionability_score_pima_diabetes/datasets/ees_actionability_model_scores.csv
Regulatory alignment model scores found at: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_05_ees_framework_regulatory_alignment_score_pima_diabetes/datasets/ees_regulatory_alignment_model_scores.csv
Final EES model scores found at: /

## 5. Validate Consolidation Inputs

In [6]:
# =========================================================
# 5. Validate Consolidation Inputs
# =========================================================

validation_rows = []

required_inputs = [
    ("baseline", baseline_df, ["model_id", "model_name"]),
    ("fidelity", fidelity_df, ["model_id", "fidelity_score"]),
    ("stability", stability_df, ["model_id", "stability_score"]),
    ("actionability", actionability_df, ["model_id", "actionability_score"]),
    ("regulatory", regulatory_df, ["model_id", "regulatory_alignment_score"]),
    ("ees", ees_df, ["model_id", "ees_score"])
]

for table_name, df, required_cols in required_inputs:
    validation_rows.append({
        "check": f"{table_name} table not empty",
        "passed": not df.empty
    })
    for col in required_cols:
        validation_rows.append({
            "check": f"{table_name} has {col}",
            "passed": col in df.columns
        })

optional_inputs = [
    ("weighting_sensitivity", weighting_sensitivity_df),
    ("ranking_robustness", ranking_robustness_df),
    ("ahp_scheme_weights", ahp_scheme_weights_df),
    ("ahp_diagnostics", ahp_diagnostics_df),
    ("ahp_pairwise_matrices", ahp_pairwise_matrices_df),
]

for table_name, df in optional_inputs:
    validation_rows.append({
        "check": f"{table_name} optional table available",
        "passed": not df.empty
    })

consolidation_input_validation_df = pd.DataFrame(validation_rows)

display(consolidation_input_validation_df)
save_df(consolidation_input_validation_df, "consolidation_input_validation.csv")

required_pass = consolidation_input_validation_df[
    ~consolidation_input_validation_df["check"].str.contains("optional")
]["passed"].all()

if not required_pass:
    raise ValueError("One or more required consolidation input validation checks failed.")

,check,passed
0,baseline table not empty,True
1,baseline has model_id,True
2,baseline has model_name,True
3,fidelity table not empty,True
4,fidelity has model_id,True
5,fidelity has fidelity_score,True
6,stability table not empty,True
7,stability has model_id,True
8,stability has stability_score,True
9,actionability table not empty,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/tables/consolidation_input_validation.csv


## 6. Build Final Consolidated Results Table

In [7]:
# =========================================================
# 6. Build Final Consolidated Results Table
# =========================================================

base_cols = safe_cols(
    baseline_df,
    [
        "model_id",
        "model_name",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "roc_auc",
        "average_precision",
        "business_priority_score",
        "business_priority_rank"
    ]
)

final_consolidated_results_df = baseline_df[base_cols].copy()

fidelity_cols = safe_cols(
    fidelity_df,
    [
        "model_id",
        "shap_reconstruction_fidelity_score",
        "lime_fidelity_score",
        "shap_lime_agreement_score",
        "fidelity_score",
        "fidelity_score_pct",
        "fidelity_rank",
        "fidelity_band"
    ]
)

final_consolidated_results_df = final_consolidated_results_df.merge(
    fidelity_df[fidelity_cols],
    on="model_id",
    how="left"
)

stability_cols = safe_cols(
    stability_df,
    [
        "model_id",
        "mean_local_lipschitz_ratio",
        "stability_score",
        "stability_score_pct",
        "stability_rank",
        "stability_band"
    ]
)

final_consolidated_results_df = final_consolidated_results_df.merge(
    stability_df[stability_cols],
    on="model_id",
    how="left"
)

actionability_cols = safe_cols(
    actionability_df,
    [
        "model_id",
        "availability_score",
        "proximity_score",
        "sparsity_score",
        "actionability_score",
        "actionability_score_pct",
        "actionability_rank",
        "actionability_band"
    ]
)

final_consolidated_results_df = final_consolidated_results_df.merge(
    actionability_df[actionability_cols],
    on="model_id",
    how="left"
)

regulatory_cols = safe_cols(
    regulatory_df,
    [
        "model_id",
        "transparency_score",
        "explanation_availability_score",
        "fidelity_alignment_score",
        "stability_alignment_score",
        "actionability_contestability_score",
        "sensitive_feature_awareness_score",
        "documentation_auditability_score",
        "regulatory_alignment_score",
        "regulatory_alignment_score_pct",
        "regulatory_alignment_rank",
        "regulatory_alignment_band"
    ]
)

final_consolidated_results_df = final_consolidated_results_df.merge(
    regulatory_df[regulatory_cols],
    on="model_id",
    how="left"
)

ees_cols = safe_cols(
    ees_df,
    [
        "model_id",
        "scheme_1_equal_weight_score",
        "scheme_1_equal_weight_score_pct",
        "scheme_1_equal_weight_rank",
        "scheme_2_fidelity_priority_score",
        "scheme_2_fidelity_priority_rank",
        "scheme_3_stability_priority_score",
        "scheme_3_stability_priority_rank",
        "scheme_4_actionability_priority_score",
        "scheme_4_actionability_priority_rank",
        "scheme_5_regulatory_priority_score",
        "scheme_5_regulatory_priority_rank",
        "scheme_6_theoretical_priority_score",
        "scheme_6_theoretical_priority_rank",
        "ees_equal_weight_score",
        "ees_equal_weight_score_pct",
        "ees_ahp_score",
        "ees_ahp_score_pct",
        "ees_score",
        "ees_score_pct",
        "ees_rank",
        "ees_band",
        "n_components_available"
    ]
)

final_consolidated_results_df = final_consolidated_results_df.merge(
    ees_df[ees_cols],
    on="model_id",
    how="left"
)

if not ranking_robustness_df.empty:
    robustness_cols = safe_cols(
        ranking_robustness_df,
        [
            "model_id",
            "mean_rank_across_schemes",
            "median_rank_across_schemes",
            "min_rank_across_schemes",
            "max_rank_across_schemes",
            "rank_range_across_schemes",
            "max_rank_change_vs_equal_weight",
            "mean_rank_change_vs_equal_weight",
            "ranking_robustness_note"
        ]
    )

    final_consolidated_results_df = final_consolidated_results_df.merge(
        ranking_robustness_df[robustness_cols],
        on="model_id",
        how="left"
    )

final_consolidated_results_df = add_dataset_metadata(final_consolidated_results_df)

final_consolidated_results_df = (
    final_consolidated_results_df
    .sort_values(["ees_rank", "model_id"])
    .reset_index(drop=True)
)

display(final_consolidated_results_df)

,dataset_name,dataset_label,model_id,model_name,accuracy,precision,recall,f1_score,roc_auc,average_precision,business_priority_score,business_priority_rank,shap_reconstruction_fidelity_score,lime_fidelity_score,shap_lime_agreement_score,fidelity_score,fidelity_score_pct,fidelity_rank,fidelity_band,mean_local_lipschitz_ratio,stability_score,stability_score_pct,stability_rank,stability_band,availability_score,proximity_score,sparsity_score,actionability_score,actionability_score_pct,actionability_rank,actionability_band,transparency_score,explanation_availability_score,sensitive_feature_awareness_score,documentation_auditability_score,regulatory_alignment_score,regulatory_alignment_score_pct,regulatory_alignment_rank,scheme_1_equal_weight_score,scheme_1_equal_weight_score_pct,scheme_1_equal_weight_rank,scheme_2_fidelity_priority_score,scheme_2_fidelity_priority_rank,scheme_3_stability_priority_score,scheme_3_stability_priority_rank,scheme_4_actionability_priority_score,scheme_4_actionability_priority_rank,scheme_5_regulatory_priority_score,scheme_5_regulatory_priority_rank,scheme_6_theoretical_priority_score,scheme_6_theoretical_priority_rank,ees_equal_weight_score,ees_equal_weight_score_pct,ees_ahp_score,ees_ahp_score_pct,ees_score,ees_score_pct,ees_rank,ees_band,n_components_available,mean_rank_across_schemes,median_rank_across_schemes,min_rank_across_schemes,max_rank_across_schemes,rank_range_across_schemes,max_rank_change_vs_equal_weight,mean_rank_change_vs_equal_weight,ranking_robustness_note
0,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,0.701299,0.586957,0.500000,0.540000,0.812222,0.673052,0.611058,5,1.000000,0.317475,0.867347,0.728274,72.827392,1,Moderate-high fidelity,0.079345,1.000000,100.000000,1,High stability,1.000000,0.943100,0.847826,0.930309,93.030868,5,High actionability,1.000000,1.000000,1.000000,1.000000,1.000000,100.000000,1,0.914646,91.464565,1,0.877371,1,0.931717,1,0.917778,1,0.931717,1,0.866373,1,0.914646,91.464565,0.866373,86.637251,0.914646,91.464565,1,High EES,4,1.000000,1.000000,1,1,0,0,0.000000,No rank change across sensitivity schemes.
1,pima_diabetes,Pima Diabetes,mlp,MLP,0.714286,0.613636,0.500000,0.551020,0.814630,0.689200,0.617116,4,1.000000,0.290141,0.794063,0.694735,69.473484,2,Moderate-high fidelity,0.087623,0.952920,95.292026,2,High stability,1.000000,0.945213,0.857955,0.934389,93.438902,4,High actionability,1.000000,1.000000,1.000000,1.000000,1.000000,100.000000,1,0.895511,89.551103,2,0.855356,2,0.906993,2,0.903287,2,0.916409,2,0.843552,2,0.895511,89.551103,0.843552,84.355231,0.895511,89.551103,2,High EES,4,2.000000,2.000000,2,2,0,0,0.000000,No rank change across sensitivity schemes.
2,pima_diabetes,Pima Diabetes,random_forest,Random Forest,0.740260,0.659091,0.537037,0.591837,0.815093,0.698897,0.644864,3,0.998148,0.276355,0.766234,0.680246,68.024573,3,Moderate-high fidelity,0.160645,0.537596,53.759597,3,Moderate stability,1.000000,0.952648,0.855114,0.935921,93.592064,3,High actionability,1.000000,1.000000,1.000000,1.000000,1.000000,100.000000,1,0.788441,78.844059,3,0.766802,3,0.738272,3,0.817937,3,0.830752,3,0.770439,3,0.788441,78.844059,0.770439,77.043915,0.788441,78.844059,3,Moderate-high EES,4,3.000000,3.000000,3,3,0,0,0.000000,No rank change across sensitivity schemes.
3,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,0.759740,0.639344,0.722222,0.678261,0.762222,0.585402,0.717647,1,1.000000,0.181610,0.744666,0.642092,64.209189,5,Moderate-high fidelity,0.166242,0.505760,50.575989,4,Moderate stability,1.000000,0.955848,0.870902,0.942250,94.224985,1,High actionability,1.000000,1.000000,1.000000,1.000000,1.000000,100.000000,1,0.772525,77.252541,4,0.746439,4,0.719172,4,0.806470,4,0.818020,4,0.748117,4,0.772525,77.252541,0.748117,74.811745,0.772525,77.252541,4,Moderate-high EES,4,4.000000,4.000000,4,4,0,0,0.000000,No rank change across sensitivity schemes.
4,pima_diabetes,Pima Diabetes,xgboost,XGBoost,0.772727,0.686275,0.648148,0.666667,0.826481,0.671049,0.709273

## 7. Create Publication-Ready Summary Table

In [8]:
# =========================================================
# 7. Create Publication-Ready Summary Table
# =========================================================

publication_cols = safe_cols(
    final_consolidated_results_df,
    [
        "dataset_name",
        "dataset_label",
        "model_id",
        "model_name",
        "accuracy",
        "precision",
        "recall",
        "roc_auc",
        "business_priority_score",
        "shap_reconstruction_fidelity_score",
        "lime_fidelity_score",
        "shap_lime_agreement_score",
        "fidelity_score",
        "stability_score",
        "availability_score",
        "proximity_score",
        "sparsity_score",
        "actionability_score",
        "regulatory_alignment_score",
        "ees_score",
        "ees_score_pct",
        "ees_rank",
        "ees_band",
        "scheme_6_theoretical_priority_score",
        "scheme_6_theoretical_priority_rank",
        "mean_rank_across_schemes",
        "max_rank_change_vs_equal_weight",
        "ranking_robustness_note"
    ]
)

publication_ready_summary_df = final_consolidated_results_df[publication_cols].copy()

display(publication_ready_summary_df)

,dataset_name,dataset_label,model_id,model_name,accuracy,precision,recall,roc_auc,business_priority_score,shap_reconstruction_fidelity_score,lime_fidelity_score,shap_lime_agreement_score,fidelity_score,stability_score,availability_score,proximity_score,sparsity_score,actionability_score,regulatory_alignment_score,ees_score,ees_score_pct,ees_rank,ees_band,scheme_6_theoretical_priority_score,scheme_6_theoretical_priority_rank,mean_rank_across_schemes,max_rank_change_vs_equal_weight,ranking_robustness_note
0,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,0.701299,0.586957,0.500000,0.812222,0.611058,1.000000,0.317475,0.867347,0.728274,1.000000,1.000000,0.943100,0.847826,0.930309,1.000000,0.914646,91.464565,1,High EES,0.866373,1,1.000000,0,No rank change across sensitivity schemes.
1,pima_diabetes,Pima Diabetes,mlp,MLP,0.714286,0.613636,0.500000,0.814630,0.617116,1.000000,0.290141,0.794063,0.694735,0.952920,1.000000,0.945213,0.857955,0.934389,1.000000,0.895511,89.551103,2,High EES,0.843552,2,2.000000,0,No rank change across sensitivity schemes.
2,pima_diabetes,Pima Diabetes,random_forest,Random Forest,0.740260,0.659091,0.537037,0.815093,0.644864,0.998148,0.276355,0.766234,0.680246,0.537596,1.000000,0.952648,0.855114,0.935921,1.000000,0.788441,78.844059,3,Moderate-high EES,0.770439,3,3.000000,0,No rank change across sensitivity schemes.
3,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,0.759740,0.639344,0.722222,0.762222,0.717647,1.000000,0.181610,0.744666,0.642092,0.505760,1.000000,0.955848,0.870902,0.942250,1.000000,0.772525,77.252541,4,Moderate-high EES,0.748117,4,4.000000,0,No rank change across sensitivity schemes.
4,pima_diabetes,Pima Diabetes,xgboost,XGBoost,0.772727,0.686275,0.648148,0.826481,0.709273,1.000000,0.252181,0.719852,0.657344,0.000000,1.000000,0.952764,0.862745,0.938503,1.000000,0.648962,64.896182,5,Moderate-high EES,0.673921,5,5.000000,0,No rank change across sensitivity schemes.


## 8. Create Model Ranking Summary

In [9]:
# =========================================================
# 8. Create Model Ranking Summary
# =========================================================

ranking_summary_df = final_consolidated_results_df[
    safe_cols(
        final_consolidated_results_df,
        [
            "dataset_name",
            "dataset_label",
            "model_id",
            "model_name",
            "business_priority_rank",
            "fidelity_rank",
            "stability_rank",
            "actionability_rank",
            "regulatory_alignment_rank",
            "ees_rank",
            "scheme_1_equal_weight_rank",
            "scheme_2_fidelity_priority_rank",
            "scheme_3_stability_priority_rank",
            "scheme_4_actionability_priority_rank",
            "scheme_5_regulatory_priority_rank",
            "scheme_6_theoretical_priority_rank",
            "mean_rank_across_schemes",
            "rank_range_across_schemes",
            "ranking_robustness_note",
            "ees_score",
            "ees_band"
        ]
    )
].copy()

display(ranking_summary_df)

,dataset_name,dataset_label,model_id,model_name,business_priority_rank,fidelity_rank,stability_rank,actionability_rank,regulatory_alignment_rank,ees_rank,scheme_1_equal_weight_rank,scheme_2_fidelity_priority_rank,scheme_3_stability_priority_rank,scheme_4_actionability_priority_rank,scheme_5_regulatory_priority_rank,scheme_6_theoretical_priority_rank,mean_rank_across_schemes,rank_range_across_schemes,ranking_robustness_note,ees_score,ees_band
0,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,5,1,1,5,1,1,1,1,1,1,1,1,1.000000,0,No rank change across sensitivity schemes.,0.914646,High EES
1,pima_diabetes,Pima Diabetes,mlp,MLP,4,2,2,4,1,2,2,2,2,2,2,2,2.000000,0,No rank change across sensitivity schemes.,0.895511,High EES
2,pima_diabetes,Pima Diabetes,random_forest,Random Forest,3,3,3,3,1,3,3,3,3,3,3,3,3.000000,0,No rank change across sensitivity schemes.,0.788441,Moderate-high EES
3,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,1,5,4,1,1,4,4,4,4,4,4,4,4.000000,0,No rank change across sensitivity schemes.,0.772525,Moderate-high EES
4,pima_diabetes,Pima Diabetes,xgboost,XGBoost,2,4,5,2,1,5,5,5,5,5,5,5,5.000000,0,No rank change across sensitivity schemes.,0.648962,Moderate-high EES


## 9. Create EES Component Score Matrix

In [10]:
# =========================================================
# 9. Create EES Component Score Matrix
# =========================================================

ees_component_score_matrix_df = final_consolidated_results_df[
    safe_cols(
        final_consolidated_results_df,
        [
            "dataset_name",
            "dataset_label",
            "model_id",
            "model_name",
            "fidelity_score",
            "stability_score",
            "actionability_score",
            "regulatory_alignment_score",
            "ees_score",
            "ees_rank"
        ]
    )
].copy()

display(ees_component_score_matrix_df)

,dataset_name,dataset_label,model_id,model_name,fidelity_score,stability_score,actionability_score,regulatory_alignment_score,ees_score,ees_rank
0,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,0.728274,1.000000,0.930309,1.000000,0.914646,1
1,pima_diabetes,Pima Diabetes,mlp,MLP,0.694735,0.952920,0.934389,1.000000,0.895511,2
2,pima_diabetes,Pima Diabetes,random_forest,Random Forest,0.680246,0.537596,0.935921,1.000000,0.788441,3
3,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,0.642092,0.505760,0.942250,1.000000,0.772525,4
4,pima_diabetes,Pima Diabetes,xgboost,XGBoost,0.657344,0.000000,0.938503,1.000000,0.648962,5


## 10. Create AHP Sensitivity Summary

This section consolidates the six AHP-derived sensitivity schemes from `08_06`.

In [11]:
# =========================================================
# 10. Create AHP Sensitivity Summary
# =========================================================

if weighting_sensitivity_df.empty:
    ees_sensitivity_summary_df = pd.DataFrame()
else:
    ees_sensitivity_summary_df = add_dataset_metadata(weighting_sensitivity_df.copy())

display(ees_sensitivity_summary_df.head(30))

,dataset_name,dataset_label,model_id,model_name,scheme_id,scheme_name,scheme_role,ees_score_under_scheme,ees_score_under_scheme_pct,rank_under_scheme,scheme_1_equal_weight_rank,absolute_rank_change_vs_equal_weight
0,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,0.772525,77.252541,4,4,0
1,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,0.746439,74.643871,4,4,0
2,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,0.719172,71.917231,4,4,0
3,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,0.806470,80.647030,4,4,0
4,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,scheme_5_regulatory_priority,Scheme 5 - Regulatory Alignment Priority,Sensitivity analysis,0.818020,81.802033,4,4,0
5,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,scheme_6_theoretical_priority,Scheme 6 - Theoretical Priority F > R > S > A,Sensitivity analysis,0.748117,74.811745,4,4,0
6,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,0.914646,91.464565,1,1,0
7,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,0.877371,87.737130,1,1,0
8,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,0.931717,93.171652,1,1,0
9,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,0.917778,91.777826,1,1,0


## 11. Create Ranking Robustness Summary

This section summarises how stable each model's rank is across all AHP-derived weighting schemes.

In [12]:
# =========================================================
# 11. Create Ranking Robustness Summary
# =========================================================

if ranking_robustness_df.empty:
    ees_ranking_robustness_summary_df = pd.DataFrame()
else:
    ees_ranking_robustness_summary_df = add_dataset_metadata(ranking_robustness_df.copy())

display(ees_ranking_robustness_summary_df)

,dataset_name,dataset_label,model_id,model_name,mean_rank_across_schemes,median_rank_across_schemes,min_rank_across_schemes,max_rank_across_schemes,rank_range_across_schemes,mean_score_across_schemes,min_score_across_schemes,max_score_across_schemes,max_rank_change_vs_equal_weight,mean_rank_change_vs_equal_weight,ranking_robustness_note
0,pima_diabetes,Pima Diabetes,logistic_regression,Logistic Regression,1.000000,1.000000,1,1,0,0.906600,0.866373,0.931717,0,0.000000,No rank change across sensitivity schemes.
1,pima_diabetes,Pima Diabetes,mlp,MLP,2.000000,2.000000,2,2,0,0.886851,0.843552,0.916409,0,0.000000,No rank change across sensitivity schemes.
2,pima_diabetes,Pima Diabetes,random_forest,Random Forest,3.000000,3.000000,3,3,0,0.785440,0.738272,0.830752,0,0.000000,No rank change across sensitivity schemes.
3,pima_diabetes,Pima Diabetes,decision_tree,Decision Tree,4.000000,4.000000,4,4,0,0.768457,0.719172,0.818020,0,0.000000,No rank change across sensitivity schemes.
4,pima_diabetes,Pima Diabetes,xgboost,XGBoost,5.000000,5.000000,5,5,0,0.653122,0.519169,0.719169,0,0.000000,No rank change across sensitivity schemes.


## 12. Create AHP Diagnostics Summary

This section records AHP consistency diagnostics for all weighting schemes.

In [13]:
# =========================================================
# 12. Create AHP Diagnostics Summary
# =========================================================

if ahp_diagnostics_df.empty:
    ahp_diagnostics_summary_df = pd.DataFrame()
else:
    ahp_diagnostics_summary_df = ahp_diagnostics_df.copy()
    ahp_diagnostics_summary_df = add_dataset_metadata(ahp_diagnostics_summary_df)

display(ahp_diagnostics_summary_df)

,dataset_name,dataset_label,scheme_id,scheme_name,scheme_role,lambda_max,consistency_index,random_index,consistency_ratio,is_consistent_under_0_10
0,pima_diabetes,Pima Diabetes,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,4.000000,-0.000000,0.900000,-0.000000,True
1,pima_diabetes,Pima Diabetes,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
2,pima_diabetes,Pima Diabetes,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
3,pima_diabetes,Pima Diabetes,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
4,pima_diabetes,Pima Diabetes,scheme_5_regulatory_priority,Scheme 5 - Regulatory Alignment Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
5,pima_diabetes,Pima Diabetes,scheme_6_theoretical_priority,Scheme 6 - Theoretical Priority F > R > S > A,Sensitivity analysis,4.030983,0.010328,0.900000,0.011475,True


## 13. Create AHP Scheme Weights Summary

In [14]:
# =========================================================
# 13. Create AHP Scheme Weights Summary
# =========================================================

if ahp_scheme_weights_df.empty:
    ahp_scheme_weights_summary_df = pd.DataFrame()
else:
    ahp_scheme_weights_summary_df = ahp_scheme_weights_df.copy()
    ahp_scheme_weights_summary_df = add_dataset_metadata(ahp_scheme_weights_summary_df)

display(ahp_scheme_weights_summary_df.head(30))

,dataset_name,dataset_label,scheme_id,scheme_name,scheme_role,pillar,pillar_column,ahp_weight,ahp_weight_pct
0,pima_diabetes,Pima Diabetes,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Fidelity,fidelity_score,0.250000,25.000000
1,pima_diabetes,Pima Diabetes,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Stability,stability_score,0.250000,25.000000
2,pima_diabetes,Pima Diabetes,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Actionability,actionability_score,0.250000,25.000000
3,pima_diabetes,Pima Diabetes,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Regulatory Alignment,regulatory_alignment_score,0.250000,25.000000
4,pima_diabetes,Pima Diabetes,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Fidelity,fidelity_score,0.400000,40.000000
5,pima_diabetes,Pima Diabetes,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Stability,stability_score,0.200000,20.000000
6,pima_diabetes,Pima Diabetes,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Actionability,actionability_score,0.200000,20.000000
7,pima_diabetes,Pima Diabetes,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Regulatory Alignment,regulatory_alignment_score,0.200000,20.000000
8,pima_diabetes,Pima Diabetes,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,Fidelity,fidelity_score,0.200000,20.000000
9,pima_diabetes,Pima Diabetes,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,Stability,stability_score,0.400000,40.000000


## 14. Create Methodology Audit Table

In [15]:
# =========================================================
# 14. Create Methodology Audit Table
# =========================================================

methodology_audit_df = pd.DataFrame([
    {
        "component": "Business Priority Score",
        "definition": "Predictive model-selection score.",
        "calculation": "0.50 Recall + 0.30 ROC-AUC + 0.20 Precision.",
        "purpose": "Champion predictive model selection."
    },
    {
        "component": "Fidelity",
        "definition": "Degree to which explanations accurately and consistently reflect model behaviour.",
        "calculation": "Mean of SHAP reconstruction fidelity, LIME local surrogate fidelity, and SHAP-LIME agreement.",
        "purpose": "EES Pillar 1."
    },
    {
        "component": "SHAP Reconstruction Fidelity",
        "definition": "Internal additive reconstruction faithfulness of SHAP explanations.",
        "calculation": "1 - mean reconstruction error.",
        "purpose": "Fidelity subcomponent."
    },
    {
        "component": "LIME Local Surrogate Fidelity",
        "definition": "Quality of LIME local surrogate approximation.",
        "calculation": "Mean LIME R² score.",
        "purpose": "Fidelity subcomponent."
    },
    {
        "component": "SHAP-LIME Agreement",
        "definition": "Cross-method explanation consistency.",
        "calculation": "Normalised Kendall Tau: (tau + 1) / 2.",
        "purpose": "Fidelity subcomponent."
    },
    {
        "component": "Stability",
        "definition": "Sensitivity of model output to small local perturbations.",
        "calculation": "Min-max normalised inverse mean local Lipschitz ratio.",
        "purpose": "EES Pillar 2."
    },
    {
        "component": "Actionability",
        "definition": "Whether unfavourable outcomes are contestable, close, and simple.",
        "calculation": "Mean of Availability, Proximity, and Sparsity.",
        "purpose": "EES Pillar 3."
    },
    {
        "component": "Availability",
        "definition": "Whether at least one counterfactual can be generated.",
        "calculation": "n_actionable_recourse_instances / n_unfavourable_instances.",
        "purpose": "Actionability subcomponent."
    },
    {
        "component": "Proximity",
        "definition": "Numerical closeness of the nearest counterfactual.",
        "calculation": "1 - mean minimum normalized numeric distance.",
        "purpose": "Actionability subcomponent."
    },
    {
        "component": "Sparsity",
        "definition": "Simplicity of counterfactual recourse.",
        "calculation": "1 - mean changed features / total features.",
        "purpose": "Actionability subcomponent."
    },
    {
        "component": "Regulatory Alignment",
        "definition": "Structured governance-readiness evidence.",
        "calculation": "Mean of seven equal-weight regulatory rubric items.",
        "purpose": "EES Pillar 4."
    },
    {
        "component": "Primary Final EES",
        "definition": "Primary composite explainability-ethics score.",
        "calculation": "AHP-derived equal weighting: 25% each for Fidelity, Stability, Actionability, Regulatory Alignment.",
        "purpose": "Main EES result."
    },
    {
        "component": "AHP Sensitivity Analysis",
        "definition": "Alternative AHP-derived pillar weightings.",
        "calculation": "Six AHP schemes: equal weight, single-pillar priorities, and theoretical F > R > S > A.",
        "purpose": "Robustness assessment."
    }
])

display(methodology_audit_df)

,component,definition,calculation,purpose
0,Business Priority Score,Predictive model-selection score.,0.50 Recall + 0.30 ROC-AUC + 0.20 Precision.,Champion predictive model selection.
1,Fidelity,Degree to which explanations accurately and co...,"Mean of SHAP reconstruction fidelity, LIME loc...",EES Pillar 1.
2,SHAP Reconstruction Fidelity,Internal additive reconstruction faithfulness ...,1 - mean reconstruction error.,Fidelity subcomponent.
3,LIME Local Surrogate Fidelity,Quality of LIME local surrogate approximation.,Mean LIME R² score.,Fidelity subcomponent.
4,SHAP-LIME Agreement,Cross-method explanation consistency.,Normalised Kendall Tau: (tau + 1) / 2.,Fidelity subcomponent.
5,Stability,Sensitivity of model output to small local per...,Min-max normalised inverse mean local Lipschit...,EES Pillar 2.
6,Actionability,"Whether unfavourable outcomes are contestable,...","Mean of Availability, Proximity, and Sparsity.",EES Pillar 3.
7,Availability,Whether at least one counterfactual can be gen...,n_actionable_recourse_instances / n_unfavourab...,Actionability subcomponent.
8,Proximity,Numerical closeness of the nearest counterfact...,1 - mean minimum normalized numeric distance.,Actionability subcomponent.
9,Sparsity,Simplicity of counterfactual recourse.,1 - mean changed features / total features.,Actionability subcomponent.


## 15. Output Inventory

In [16]:
# =========================================================
# 15. Output Inventory
# =========================================================

output_inventory_df = pd.DataFrame([
    {
        "output_file": "final_consolidated_results.csv",
        "description": "Full model-level consolidated results table.",
        "primary_use": "Dissertation results chapter and final comparison."
    },
    {
        "output_file": "publication_ready_summary.csv",
        "description": "Compact table for journal article reporting.",
        "primary_use": "ESWA manuscript tables."
    },
    {
        "output_file": "model_ranking_summary.csv",
        "description": "Model ranking comparison across performance, EES pillars, and AHP schemes.",
        "primary_use": "Discussion and interpretation."
    },
    {
        "output_file": "ees_component_score_matrix.csv",
        "description": "Side-by-side EES component scores.",
        "primary_use": "Framework analysis and visualisation."
    },
    {
        "output_file": "ees_sensitivity_summary.csv",
        "description": "Long-format AHP sensitivity results across all weighting schemes.",
        "primary_use": "Weighting robustness analysis."
    },
    {
        "output_file": "ees_ranking_robustness_summary.csv",
        "description": "Rank stability summary across AHP-derived schemes.",
        "primary_use": "Methodology robustness evidence."
    },
    {
        "output_file": "ahp_diagnostics_summary.csv",
        "description": "AHP consistency diagnostics by weighting scheme.",
        "primary_use": "AHP audit and methodological defence."
    },
    {
        "output_file": "ahp_scheme_weights_summary.csv",
        "description": "AHP-derived pillar weights for each scheme.",
        "primary_use": "AHP sensitivity reporting."
    },
    {
        "output_file": "methodology_audit_table.csv",
        "description": "Updated scoring method definitions.",
        "primary_use": "Methodology and reproducibility documentation."
    },
    {
        "output_file": "results_consolidation_<dataset>_outputs.xlsx",
        "description": "Excel workbook containing all consolidated outputs.",
        "primary_use": "Supervisor review and reporting."
    }
])

display(output_inventory_df)

,output_file,description,primary_use
0,final_consolidated_results.csv,Full model-level consolidated results table.,Dissertation results chapter and final compari...
1,publication_ready_summary.csv,Compact table for journal article reporting.,ESWA manuscript tables.
2,model_ranking_summary.csv,"Model ranking comparison across performance, E...",Discussion and interpretation.
3,ees_component_score_matrix.csv,Side-by-side EES component scores.,Framework analysis and visualisation.
4,ees_sensitivity_summary.csv,Long-format AHP sensitivity results across all...,Weighting robustness analysis.
5,ees_ranking_robustness_summary.csv,Rank stability summary across AHP-derived sche...,Methodology robustness evidence.
6,ahp_diagnostics_summary.csv,AHP consistency diagnostics by weighting scheme.,AHP audit and methodological defence.
7,ahp_scheme_weights_summary.csv,AHP-derived pillar weights for each scheme.,AHP sensitivity reporting.
8,methodology_audit_table.csv,Updated scoring method definitions.,Methodology and reproducibility documentation.
9,results_consolidation_<dataset>_outputs.xlsx,Excel workbook containing all consolidated out...,Supervisor review and reporting.


## 16. Save Consolidated Outputs

In [17]:
# =========================================================
# 16. Save Consolidated Outputs
# =========================================================

save_df(final_consolidated_results_df, "final_consolidated_results.csv")
save_df(publication_ready_summary_df, "publication_ready_summary.csv")
save_df(ranking_summary_df, "model_ranking_summary.csv")
save_df(ees_component_score_matrix_df, "ees_component_score_matrix.csv")
save_df(ees_sensitivity_summary_df, "ees_sensitivity_summary.csv")
save_df(ees_ranking_robustness_summary_df, "ees_ranking_robustness_summary.csv")
save_df(ahp_diagnostics_summary_df, "ahp_diagnostics_summary.csv")
save_df(ahp_scheme_weights_summary_df, "ahp_scheme_weights_summary.csv")
save_df(ahp_pairwise_matrices_df, "ahp_pairwise_matrices.csv")
save_df(ahp_scheme_metadata_df, "ahp_scheme_metadata.csv")
save_df(methodology_audit_df, "methodology_audit_table.csv")
save_df(output_inventory_df, "results_consolidation_output_inventory.csv")

save_df(final_consolidated_results_df, "final_consolidated_results.csv", folder=DATASETS_PATH)
save_df(publication_ready_summary_df, "publication_ready_summary.csv", folder=DATASETS_PATH)
save_df(ranking_summary_df, "model_ranking_summary.csv", folder=DATASETS_PATH)
save_df(ees_component_score_matrix_df, "ees_component_score_matrix.csv", folder=DATASETS_PATH)
save_df(ees_sensitivity_summary_df, "ees_sensitivity_summary.csv", folder=DATASETS_PATH)
save_df(ees_ranking_robustness_summary_df, "ees_ranking_robustness_summary.csv", folder=DATASETS_PATH)
save_df(ahp_diagnostics_summary_df, "ahp_diagnostics_summary.csv", folder=DATASETS_PATH)
save_df(ahp_scheme_weights_summary_df, "ahp_scheme_weights_summary.csv", folder=DATASETS_PATH)
save_df(ahp_pairwise_matrices_df, "ahp_pairwise_matrices.csv", folder=DATASETS_PATH)
save_df(ahp_scheme_metadata_df, "ahp_scheme_metadata.csv", folder=DATASETS_PATH)
save_df(methodology_audit_df, "methodology_audit_table.csv", folder=DATASETS_PATH)
save_df(output_inventory_df, "results_consolidation_output_inventory.csv", folder=DATASETS_PATH)

save_excel(
    {
        "final_consolidated": final_consolidated_results_df,
        "publication_summary": publication_ready_summary_df,
        "ranking_summary": ranking_summary_df,
        "component_matrix": ees_component_score_matrix_df,
        "sensitivity_summary": ees_sensitivity_summary_df,
        "ranking_robustness": ees_ranking_robustness_summary_df,
        "ahp_diagnostics": ahp_diagnostics_summary_df,
        "ahp_scheme_weights": ahp_scheme_weights_summary_df,
        "ahp_pairwise": ahp_pairwise_matrices_df,
        "ahp_scheme_metadata": ahp_scheme_metadata_df,
        "component_contributions": component_contribution_df,
        "methodology_audit": methodology_audit_df,
        "input_validation": consolidation_input_validation_df,
        "output_inventory": output_inventory_df
    },
    f"results_consolidation_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "consolidated_components": [
        "baseline performance",
        "fidelity",
        "stability",
        "actionability",
        "regulatory alignment",
        "final EES",
        "AHP sensitivity",
        "ranking robustness",
        "AHP consistency diagnostics"
    ],
    "updated_fidelity_design": "SHAP reconstruction, LIME local surrogate fidelity, and SHAP-LIME agreement.",
    "updated_actionability_design": "Availability, proximity, and sparsity.",
    "final_ees_design": "Primary AHP-derived equal-weight four-pillar EES score.",
    "ahp_sensitivity_design": "Six AHP-derived weighting schemes including theoretical F > R > S > A.",
    "standardisation_note": (
        "This results consolidation notebook uses the same configurable output consolidation logic "
        "across German Credit, Pima Diabetes, and COMPAS."
    )
}

with open(LOGS_PATH / "results_consolidation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "results_consolidation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All consolidated results saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/tables/final_consolidated_results.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/tables/publication_ready_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/tables/model_ranking_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/tables/ees_component_score_matrix.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/tables/ees_sensitivity_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/tables/ees_ranking_robustness_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_

## 17. Interpretation Note

This notebook is the final dataset-level consolidation point for the EES framework.

It now includes the updated AHP sensitivity and ranking robustness outputs from `08_06`.

The most important interpretation outputs are:

- primary EES ranking under Scheme 1 equal weighting;
- ranking changes under alternative AHP-derived weighting schemes;
- consistency diagnostics for each AHP matrix;
- model-level robustness of conclusions across weighting assumptions.

For dissertation and publication reporting, use:

1. `publication_ready_summary.csv` for the main results table;
2. `ees_ranking_robustness_summary.csv` for weighting sensitivity discussion;
3. `ahp_diagnostics_summary.csv` to defend AHP consistency;
4. `methodology_audit_table.csv` for methods documentation.

In [18]:
# =========================================================
# 17. Final Completion Message
# =========================================================

print("=" * 100)
print(f"08_07 Results Consolidation completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("\nKey outputs:")
print("- final_consolidated_results.csv")
print("- publication_ready_summary.csv")
print("- model_ranking_summary.csv")
print("- ees_component_score_matrix.csv")
print("- ees_sensitivity_summary.csv")
print("- ees_ranking_robustness_summary.csv")
print("- ahp_diagnostics_summary.csv")
print("- ahp_scheme_weights_summary.csv")
print("- methodology_audit_table.csv")
print(f"- results_consolidation_{DATASET_NAME}_outputs.xlsx")

08_07 Results Consolidation completed successfully for: Pima Diabetes
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/08_07_results_consolidation_pima_diabetes/logs

Key outputs:
- final_consolidated_results.csv
- publication_ready_summary.csv
- model_ranking_summary.csv
- ees_component_score_matrix.csv
- ees_sensitivity_summary.csv
- ees_ranking_robustness_summary.csv
- ahp_diagnostics_summary.csv
- ahp_scheme_weights_summary.csv
- methodology_audit_table.csv
- results_consolidation_pima_diabetes_outputs.xlsx
